# GVH Diagonal Cubic — RACC-C.4-real.2.12
## GW150914 H1/L1 — Controlled Preprocessing Inter-Estimator Convergence Audit

**Auteur :** Charlemagne O Laurince  
**Étape :** `RACC-C.4-real.2.12`  
**Noyau GVH Diagonal modifié : NON**

### Question centrale
Après `real.2.8 → real.2.11`, tester si la divergence historique entre
\(\tau_{xcorr}\), \(\tau_{GCC-PHAT}\) et \(\tau_{phase-slope}\) se réduit lorsque phase, amplitude et poids fréquentiels sont contrôlés séparément.

Aucune valeur cible de lag n'est utilisée.

\[
\Delta\tau_{estimators}=\max_i\tau_i-\min_i\tau_i.
\]

Stop-rule :
\[
\texttt{TIMING-VALIDATED=False},\quad D_T^{ref}=\texttt{NOT-AUTHORIZED},\quad \texttt{REAL3-AUTHORIZED=False}.
\]


In [1]:
import os,sys,subprocess,importlib,json
from pathlib import Path

def pip_install(pkg): subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
try: import lalframe
except Exception:
    pip_install("lalsuite"); importlib.invalidate_caches(); import lalframe
try: import gwpy
except Exception:
    pip_install("gwpy"); importlib.invalidate_caches(); import gwpy
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import signal
from gwpy.timeseries import TimeSeries
print("gwpy:",gwpy.__version__)

/usr/local/lib/python3.12/dist-packages/lalframe/_lalframe_swig.py:8: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


gwpy: 4.0.1


## 1 — Chargement H1/L1

In [2]:
GPS_EVENT=1126259462.0; FS_EXPECTED=4096.0

def discover(pattern):
    roots=[Path.cwd(),Path('/content'),Path('/content/drive/MyDrive'),Path('/mnt/data')]
    hits=[]
    for root in roots:
        if not root.exists(): continue
        for d in [root,root/'data'/'GW150914',root/'gvh_diagonal_cubic'/'data'/'GW150914']:
            if d.exists(): hits+=list(d.glob(pattern))
    return hits[0].resolve() if hits else None
H1=discover('H-H1_LOSC_4_V2-1126259446-32*.gwf'); L1=discover('L-L1_LOSC_4_V2-1126259446-32*.gwf')
if H1 is None or L1 is None:
    try:
        from google.colab import files; files.upload()
        H1=discover('H-H1_LOSC_4_V2-1126259446-32*.gwf'); L1=discover('L-L1_LOSC_4_V2-1126259446-32*.gwf')
    except Exception: pass
if H1 is None or L1 is None: raise FileNotFoundError('Fichiers H1/L1 introuvables.')
h1_ts=TimeSeries.read(str(H1),channel='H1:LOSC-STRAIN'); l1_ts=TimeSeries.read(str(L1),channel='L1:LOSC-STRAIN')
h1=np.asarray(h1_ts.value,float); l1=np.asarray(l1_ts.value,float)
fs=float(h1_ts.sample_rate.value); gps0=float(h1_ts.t0.value)
t_rel=gps0+np.arange(len(h1))/fs-GPS_EVENT
assert np.isclose(fs,FS_EXPECTED) and len(h1)==len(l1)
print(H1.name); print(L1.name); print('N =',len(h1),'fs =',fs)

Saving L-L1_LOSC_4_V2-1126259446-32.gwf to L-L1_LOSC_4_V2-1126259446-32.gwf
Saving H-H1_LOSC_4_V2-1126259446-32 2.gwf to H-H1_LOSC_4_V2-1126259446-32 2.gwf
H-H1_LOSC_4_V2-1126259446-32 2.gwf
L-L1_LOSC_4_V2-1126259446-32.gwf
N = 131072 fs = 4096.0


## 2 — Prétraitements contrôlés

In [3]:
EVENT=(-.16,.08); EM=(t_rel>=EVENT[0])&(t_rel<=EVENT[1]); NOISE=((t_rel>=-12)&(t_rel<=-2))|((t_rel>=2)&(t_rel<=12))
def center(x): return np.asarray(x,float)-np.mean(x)
def estimate_psd(x):
    f,P=signal.welch(center(x)[NOISE],fs=fs,nperseg=4096,noverlap=2048,window='hann',detrend='constant',scaling='density'); return f,P
def whiten_with_psd(x,fp,P):
    x=center(x); n=len(x); f=np.fft.rfftfreq(n,1/fs); X=np.fft.rfft(x)
    Pi=np.interp(f,fp,P,left=P[0],right=P[-1]); sc=np.median(Pi[(f>=40)&(f<=300)]); floor=max(sc*1e-12,np.finfo(float).tiny)
    return np.fft.irfft(X/np.sqrt(np.maximum(Pi,floor)),n=n)
def band_notch(x):
    sos=signal.butter(4,[35,350],btype='bandpass',fs=fs,output='sos'); y=signal.sosfiltfilt(sos,center(x))
    for f0 in [60,120,180,240,300]:
        b,a=signal.iirnotch(f0,Q=30,fs=fs); y=signal.filtfilt(b,a,y)
    return y
fH,PH=estimate_psd(h1); fL,PL=estimate_psd(l1); fC=fH; PLi=np.interp(fC,fL,PL); PC=np.sqrt(np.maximum(PH,0)*np.maximum(PLi,0))
H_raw=band_notch(h1)[EM]; L_raw=band_notch(l1)[EM]
H_white=band_notch(whiten_with_psd(h1,fH,PH))[EM]; L_white=band_notch(whiten_with_psd(l1,fL,PL))[EM]
H_common=band_notch(whiten_with_psd(h1,fC,PC))[EM]; L_common=band_notch(whiten_with_psd(l1,fC,PC))[EM]
print('event samples:',len(H_raw))

event samples: 983


## 3 — Estimateurs

In [4]:
MAX_LAG=.012
def parab(a,b,c):
    d=a-2*b+c; return 0.0 if d==0 or not np.isfinite(d) else .5*(a-c)/d
def xcorr_lag(a,b):
    a=center(a); b=center(b); cc=signal.correlate(a,b,mode='full',method='fft'); lag=signal.correlation_lags(len(a),len(b),mode='full')/fs
    m=np.abs(lag)<=MAX_LAG; cc=cc[m]; lag=lag[m]; cn=cc/(np.linalg.norm(a)*np.linalg.norm(b)+np.finfo(float).eps); s=np.abs(cn); i=int(np.argmax(s)); frac=parab(s[i-1],s[i],s[i+1]) if 0<i<len(s)-1 else 0
    return float(lag[i]+frac/fs),float(cn[i])
def phat_lag(a,b):
    a=center(a); b=center(b); n=int(2**np.ceil(np.log2(len(a)+len(b)))); A=np.fft.rfft(a,n=n); B=np.fft.rfft(b,n=n); G=A*np.conj(B); R=G/(np.abs(G)+np.finfo(float).eps)
    cc=np.fft.irfft(R,n=n); cc=np.concatenate((cc[-n//2:],cc[:n//2])); lag=np.arange(-n//2,n//2)/fs; m=np.abs(lag)<=MAX_LAG; cc=cc[m]; lag=lag[m]; s=np.abs(cc); i=int(np.argmax(s)); frac=parab(s[i-1],s[i],s[i+1]) if 0<i<len(s)-1 else 0
    return float(lag[i]+frac/fs),float(cc[i])
def phase_objects(H,L,nper=512):
    f,P=signal.csd(L,H,fs=fs,nperseg=nper,noverlap=nper//2,window='hann'); _,coh=signal.coherence(H,L,fs=fs,nperseg=nper,noverlap=nper//2,window='hann'); ph=np.angle(P); m=(f>=40)&(f<=300)&np.isfinite(ph)&np.isfinite(coh); return f[m],ph[m],coh[m]
def fit_phase(f,ph,w,mask=None):
    if mask is None: mask=np.ones_like(f,dtype=bool)
    ff=f[mask]; pp=np.unwrap(ph[mask]); ww=np.clip(w[mask],1e-12,None)
    if len(ff)<5: return np.nan,np.nan
    A=np.c_[ff,np.ones_like(ff)]; W=np.sqrt(ww); beta,*_=np.linalg.lstsq(A*W[:,None],pp*W,rcond=None); pred=A@beta; mean=np.average(pp,weights=ww); ssr=np.sum(ww*(pp-pred)**2); sst=np.sum(ww*(pp-mean)**2); r2=1-ssr/sst if sst>0 else np.nan
    return float(beta[0]/(2*np.pi)),float(r2)

## 4 — Contrôle phase / poids

In [5]:
fR,phR,cohR=phase_objects(H_raw,L_raw); fW,phW,cohW=phase_objects(H_white,L_white); fC2,phC2,cohC2=phase_objects(H_common,L_common)
phW_i=np.interp(fR,fW,phW); cohW_i=np.interp(fR,fW,cohW); phC_i=np.interp(fR,fC2,phC2); cohC_i=np.interp(fR,fC2,cohC2)
COMMON_MASK=(cohR>=.10)&(cohW_i>=.10)&(cohC_i>=.10)
print('common bins:',int(COMMON_MASK.sum()))

common bins: 28


## 5 — Amplitude normalisée

In [6]:
def amp_norm(x):
    x=center(x); X=np.fft.rfft(x); a=np.abs(X); sc=np.median(a[a>0]) if np.any(a>0) else 1.0; return np.fft.irfft(X/(a+np.finfo(float).eps)*sc,n=len(x))
H_amp=amp_norm(H_raw); L_amp=amp_norm(L_raw)

## 6 — Scénarios contrôlés

In [7]:
rows=[]
def add(name,H,L,pf,ph,w):
    tx,qx=xcorr_lag(H,L); tp,qp=phat_lag(H,L)
    if not np.array_equal(pf,fR): ph=np.interp(fR,pf,ph); w=np.interp(fR,pf,w)
    tf,r2=fit_phase(fR,ph,w,COMMON_MASK)
    rows.extend([{'scenario':name,'estimator':'xcorr','lag_ms':tx*1e3,'quality':abs(qx)}, {'scenario':name,'estimator':'GCC-PHAT','lag_ms':tp*1e3,'quality':abs(qp)}, {'scenario':name,'estimator':'phase-slope','lag_ms':tf*1e3,'quality':r2}])
add('RAW_CONTROLLED',H_raw,L_raw,fR,phR,cohR); add('WHITE_FULL',H_white,L_white,fW,phW,cohW); add('COMMON_PSD_WHITE',H_common,L_common,fC2,phC2,cohC2); add('AMPLITUDE_NORMALIZED',H_amp,L_amp,fR,phR,cohR)
trw,r2rw=fit_phase(fR,phR,cohW_i,COMMON_MASK); twr,r2wr=fit_phase(fR,phW_i,cohR,COMMON_MASK)
txr,qxr=xcorr_lag(H_raw,L_raw); tpr,qpr=phat_lag(H_raw,L_raw); txw,qxw=xcorr_lag(H_white,L_white); tpw,qpw=phat_lag(H_white,L_white)
rows += [
 {'scenario':'RAW_PHASE_WHITE_WEIGHTS','estimator':'xcorr','lag_ms':txr*1e3,'quality':abs(qxr)}, {'scenario':'RAW_PHASE_WHITE_WEIGHTS','estimator':'GCC-PHAT','lag_ms':tpr*1e3,'quality':abs(qpr)}, {'scenario':'RAW_PHASE_WHITE_WEIGHTS','estimator':'phase-slope','lag_ms':trw*1e3,'quality':r2rw},
 {'scenario':'WHITE_PHASE_RAW_WEIGHTS','estimator':'xcorr','lag_ms':txw*1e3,'quality':abs(qxw)}, {'scenario':'WHITE_PHASE_RAW_WEIGHTS','estimator':'GCC-PHAT','lag_ms':tpw*1e3,'quality':abs(qpw)}, {'scenario':'WHITE_PHASE_RAW_WEIGHTS','estimator':'phase-slope','lag_ms':twr*1e3,'quality':r2wr}]
results=pd.DataFrame(rows); display(results)

,scenario,estimator,lag_ms,quality
0,RAW_CONTROLLED,xcorr,8.982581,1.839291e-24
1,RAW_CONTROLLED,GCC-PHAT,8.982581,1.839291e-24
2,RAW_CONTROLLED,phase-slope,0.818319,6.312941e-02
3,WHITE_FULL,xcorr,-11.962891,1.567730e-01
4,WHITE_FULL,GCC-PHAT,0.000912,6.914522e-01
5,WHITE_FULL,phase-slope,1.433352,2.460741e-01
6,COMMON_PSD_WHITE,xcorr,-10.256891,1.606965e-01
7,COMMON_PSD_WHITE,GCC-PHAT,0.011796,7.187992e-01
8,COMMON_PSD_WHITE,phase-slope,1.436904,2.412417e-01
9,AMPLITUDE_NORMALIZED,xcorr,8.982789,3.291407e-35


## 7 — Spread inter-estimateurs

In [8]:
sr=[]
for name,d in results.groupby('scenario'):
    z=d.lag_ms.to_numpy(float)
    sr.append({'scenario':name,'spread_ms':float(np.nanmax(z)-np.nanmin(z)),'xcorr_ms':float(d[d.estimator=='xcorr'].lag_ms.iloc[0]),'phat_ms':float(d[d.estimator=='GCC-PHAT'].lag_ms.iloc[0]),'phase_ms':float(d[d.estimator=='phase-slope'].lag_ms.iloc[0]),'median_quality':float(np.nanmedian(d.quality))})
spread_df=pd.DataFrame(sr).sort_values('spread_ms'); display(spread_df)

,scenario,spread_ms,xcorr_ms,phat_ms,phase_ms,median_quality
2,RAW_CONTROLLED,8.164262,8.982581,8.982581,0.818319,1.839291e-24
0,AMPLITUDE_NORMALIZED,8.164470,8.982789,8.982789,0.818319,3.291407e-35
3,RAW_PHASE_WHITE_WEIGHTS,8.866309,8.982581,8.982581,0.116272,1.839291e-24
1,COMMON_PSD_WHITE,11.693795,-10.256891,0.011796,1.436904,2.412417e-01
5,WHITE_PHASE_RAW_WEIGHTS,13.321835,-11.962891,0.000912,1.358944,1.984334e-01
4,WHITE_FULL,13.396243,-11.962891,0.000912,1.433352,2.460741e-01


## 8 — Robustesse fenêtres et bandes

In [9]:
WINDOWS=[(-.16,.04),(-.14,.04),(-.12,.04),(-.10,.04),(-.16,.06),(-.14,.06)]
wr=[]
for t0,t1 in WINDOWS:
    m=(t_rel>=t0)&(t_rel<=t1); Hr=band_notch(h1)[m]; Lr=band_notch(l1)[m]; Hw=band_notch(whiten_with_psd(h1,fH,PH))[m]; Lw=band_notch(whiten_with_psd(l1,fL,PL))[m]
    vals=[]
    for rep,H,L in [('RAW',Hr,Lr),('WHITE',Hw,Lw)]:
        tx,_=xcorr_lag(H,L); tp,_=phat_lag(H,L); f,ph,coh=phase_objects(H,L); tf,_=fit_phase(f,ph,coh,coh>=.10); vals.append({'window':f'{t0:.2f}:{t1:.2f}','representation':rep,'spread_ms':float((np.nanmax([tx,tp,tf])-np.nanmin([tx,tp,tf]))*1e3)})
    wr+=vals
window_df=pd.DataFrame(wr); display(window_df)
BANDS=[(40,100),(60,140),(80,180),(100,220),(120,260),(140,300)]; br=[]
for lo,hi in BANDS:
    def fb(x):
        sos=signal.butter(4,[lo,hi],btype='bandpass',fs=fs,output='sos'); return signal.sosfiltfilt(sos,x)
    for rep,H,L in [('RAW',fb(H_raw),fb(L_raw)),('WHITE',fb(H_white),fb(L_white))]:
        tx,_=xcorr_lag(H,L); tp,_=phat_lag(H,L); f,ph,coh=phase_objects(H,L); mm=(f>=lo)&(f<=hi)&(coh>=.10); tf,_=fit_phase(f,ph,coh,mm); br.append({'band':f'{lo}-{hi}','representation':rep,'spread_ms':float((np.nanmax([tx,tp,tf])-np.nanmin([tx,tp,tf]))*1e3)})
band_df=pd.DataFrame(br); display(band_df)

,window,representation,spread_ms
0,-0.16:0.04,RAW,8.666612
1,-0.16:0.04,WHITE,10.398756
2,-0.14:0.04,RAW,14.452042
3,-0.14:0.04,WHITE,5.521610
4,-0.12:0.04,RAW,8.455261
5,-0.12:0.04,WHITE,7.391248
6,-0.10:0.04,RAW,5.152743
7,-0.10:0.04,WHITE,7.398561
8,-0.16:0.06,RAW,8.683966
9,-0.16:0.06,WHITE,11.986803


,band,representation,spread_ms
0,40-100,RAW,6.232236
1,40-100,WHITE,3.952424
2,60-140,RAW,7.967698
3,60-140,WHITE,6.727307
4,80-180,RAW,14.832436
5,80-180,WHITE,12.494878
6,100-220,RAW,12.210571
7,100-220,WHITE,11.738748
8,120-260,RAW,8.865533
9,120-260,WHITE,10.213771


## 9 — Verdict

In [10]:
BEST=spread_df.iloc[0]; BEST_SCENARIO=str(BEST.scenario); BEST_SPREAD_MS=float(BEST.spread_ms); RAW_SPREAD_MS=float(spread_df.loc[spread_df.scenario=='RAW_CONTROLLED','spread_ms'].iloc[0]); IMP=(RAW_SPREAD_MS-BEST_SPREAD_MS)/RAW_SPREAD_MS if RAW_SPREAD_MS>0 else np.nan
WRAW=float(np.nanmedian(window_df[window_df.representation=='RAW'].spread_ms)); WW=float(np.nanmedian(window_df[window_df.representation=='WHITE'].spread_ms)); BRAW=float(np.nanmedian(band_df[band_df.representation=='RAW'].spread_ms)); BW=float(np.nanmedian(band_df[band_df.representation=='WHITE'].spread_ms))
HINT=bool(np.isfinite(IMP) and IMP>=.30 and BEST_SPREAD_MS<=5.0); ROBUST=bool(min(WRAW,WW)<=5.0 or min(BRAW,BW)<=5.0)
FINITE=bool(np.isfinite(results[['lag_ms','quality']].to_numpy()).all())
if not FINITE: FINAL_STATUS='UNRESOLVED'
elif HINT and ROBUST: FINAL_STATUS='CONTROLLED-PREPROCESSING-CONVERGENCE-HINT'
elif HINT: FINAL_STATUS='CONTROLLED-CONVERGENCE-NOT-ROBUST'
else: FINAL_STATUS='ESTIMATOR-DIVERGENCE-PERSISTS'
artifact={'step':'RACC-C.4-real.2.12','event':'GW150914','best_scenario':BEST_SCENARIO,'best_spread_ms':BEST_SPREAD_MS,'raw_spread_ms':RAW_SPREAD_MS,'improvement_fraction':IMP,'window_median_spread_raw_ms':WRAW,'window_median_spread_white_ms':WW,'band_median_spread_raw_ms':BRAW,'band_median_spread_white_ms':BW,'CONTROLLED_CONVERGENCE_HINT':HINT,'ROBUSTNESS_SUPPORT':ROBUST,'final_status':FINAL_STATUS,'TIMING_VALIDATED':False,'D_T_ref':'NOT-AUTHORIZED','REAL3_AUTHORIZED':False,'DISPERSION_READY_modified':False,'HH_chain_modified':False,'core_GVH_modified':False}
print(json.dumps(artifact,indent=2))

{
  "step": "RACC-C.4-real.2.12",
  "event": "GW150914",
  "best_scenario": "RAW_CONTROLLED",
  "best_spread_ms": 8.164262242257426,
  "raw_spread_ms": 8.164262242257426,
  "improvement_fraction": 0.0,
  "window_median_spread_raw_ms": 8.560936905580212,
  "window_median_spread_white_ms": 8.898658354188816,
  "band_median_spread_raw_ms": 10.036062264210077,
  "band_median_spread_white_ms": 10.798069278920348,
  "CONTROLLED_CONVERGENCE_HINT": false,
  "ROBUSTNESS_SUPPORT": false,
  "final_status": "ESTIMATOR-DIVERGENCE-PERSISTS",
  "TIMING_VALIDATED": false,
  "D_T_ref": "NOT-AUTHORIZED",
  "REAL3_AUTHORIZED": false,
  "DISPERSION_READY_modified": false,
  "HH_chain_modified": false,
  "core_GVH_modified": false
}


## 10 — Stop-rule finale

Même si `CONTROLLED-PREPROCESSING-CONVERGENCE-HINT` est obtenu, ce notebook ne valide pas le timing physique. Un test final indépendant, avec preprocessing gelé avant analyse, sera requis avant toute autorisation de `real.3`.
